In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

class MNISTDataset(Dataset):
  def __init__(self, path):
    data = pd.read_csv(path, header=None).to_numpy()
    self.x = data[:, 1:]
    self.y = data[:, 0]

  def __len__(self):
    return self.x.shape[0]

  def __getitem__(self, idx):
    xi = torch.tensor(self.x[idx], dtype=torch.float32)
    yi = torch.tensor(self.y[idx], dtype=torch.long)
    return xi, yi

ds = MNISTDataset('/content/sample_data/mnist_train_small.csv')
loader = DataLoader(ds, batch_size=500, shuffle=True)

for i, (bx, by) in enumerate(loader):
  print(i, bx.shape)

0 torch.Size([500, 784])
1 torch.Size([500, 784])
2 torch.Size([500, 784])
3 torch.Size([500, 784])
4 torch.Size([500, 784])
5 torch.Size([500, 784])
6 torch.Size([500, 784])
7 torch.Size([500, 784])
8 torch.Size([500, 784])
9 torch.Size([500, 784])
10 torch.Size([500, 784])
11 torch.Size([500, 784])
12 torch.Size([500, 784])
13 torch.Size([500, 784])
14 torch.Size([500, 784])
15 torch.Size([500, 784])
16 torch.Size([500, 784])
17 torch.Size([500, 784])
18 torch.Size([500, 784])
19 torch.Size([500, 784])
20 torch.Size([500, 784])
21 torch.Size([500, 784])
22 torch.Size([500, 784])
23 torch.Size([500, 784])
24 torch.Size([500, 784])
25 torch.Size([500, 784])
26 torch.Size([500, 784])
27 torch.Size([500, 784])
28 torch.Size([500, 784])
29 torch.Size([500, 784])
30 torch.Size([500, 784])
31 torch.Size([500, 784])
32 torch.Size([500, 784])
33 torch.Size([500, 784])
34 torch.Size([500, 784])
35 torch.Size([500, 784])
36 torch.Size([500, 784])
37 torch.Size([500, 784])
38 torch.Size([500, 78

In [ ]:
#Build the model
import torch.nn as nn

class MyModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(784, 100),
        nn.ReLU(),
        nn.Linear(100, 50),
        nn.ReLU(),
        nn.Linear(50, 25),
        nn.ReLU(),
        nn.Linear(25, 10)
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
#Train the model
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MyModel()
model.to(device) #move to GPU
optimizer = optim.Adam(model.parameters(), lr=0.001)
cost_func = nn.CrossEntropyLoss()

ds_train = MNISTDataset('/content/sample_data/mnist_train_small.csv')
loader_train = DataLoader(ds_train, batch_size=200, shuffle=True)

num_epochs = 50

for epoch in range(num_epochs):
  for i, (bx, by) in enumerate(loader_train):
    bx = bx.to(device)
    by = by.to(device)
    output = model(bx)
    cost = cost_func(output, by)
    if i==0:
      print('Epoch: %d, Cost: %f' % (epoch, cost.item()))
    cost.backward()
    optimizer.step()
    optimizer.zero_grad()


Epoch: 0, Cost: 4.299648
Epoch: 1, Cost: 0.283631
Epoch: 2, Cost: 0.187054
Epoch: 3, Cost: 0.107158
Epoch: 4, Cost: 0.051902
Epoch: 5, Cost: 0.081706
Epoch: 6, Cost: 0.037172
Epoch: 7, Cost: 0.030506
Epoch: 8, Cost: 0.025246
Epoch: 9, Cost: 0.037054
Epoch: 10, Cost: 0.025758
Epoch: 11, Cost: 0.016535
Epoch: 12, Cost: 0.004458
Epoch: 13, Cost: 0.007997
Epoch: 14, Cost: 0.037935
Epoch: 15, Cost: 0.016928
Epoch: 16, Cost: 0.021273
Epoch: 17, Cost: 0.016712
Epoch: 18, Cost: 0.000459
Epoch: 19, Cost: 0.009989
Epoch: 20, Cost: 0.000690
Epoch: 21, Cost: 0.013127
Epoch: 22, Cost: 0.007893
Epoch: 23, Cost: 0.007766
Epoch: 24, Cost: 0.027969
Epoch: 25, Cost: 0.014131
Epoch: 26, Cost: 0.010420
Epoch: 27, Cost: 0.038273
Epoch: 28, Cost: 0.005699
Epoch: 29, Cost: 0.001115
Epoch: 30, Cost: 0.003008
Epoch: 31, Cost: 0.002710
Epoch: 32, Cost: 0.000522
Epoch: 33, Cost: 0.000416
Epoch: 34, Cost: 0.000208
Epoch: 35, Cost: 0.000088
Epoch: 36, Cost: 0.000060
Epoch: 37, Cost: 0.000067
Epoch: 38, Cost: 0.000

In [ ]:
#Evaluate the model
ds_test = MNISTDataset('/content/sample_data/mnist_test.csv')
loader_test = DataLoader(ds_test, batch_size=200)

num_correct = 0
for bx, by in loader_test:
  bx = bx.to(device)
  by = by.to(device)
  output = model(bx)
  predict = torch.argmax(output, dim=1)
  num_correct += torch.sum(predict==by).item()

print('Accuracy: %f' % (num_correct/len(ds_test)))

Accuracy: 0.970700
